# Random sample for handcoding
Draw a random sample of rulings from `complete_data_set7May2024.csv`, keeping only those for which a summary exists in `tc_resolutions_summary.csv`.

In [ ]:
import pandas as pd

SAMPLE_SIZE = 100   # adjust as needed
RANDOM_SEED = 42

## 1. Load and deduplicate the main dataset

In [ ]:
# Long format — one row per judge per ruling. Collapse to one row per ruling.
main = pd.read_csv("complete_data_set7May2024.csv")

rulings = (
    main[["ID_PAT", "procedure", "year_r"]]
    .drop_duplicates(subset="ID_PAT")
    .reset_index(drop=True)
)
print(f"Unique rulings in main dataset: {len(rulings):,}")
rulings.head()

## 2. Load full texts

In [ ]:
fulltexts = pd.read_csv("tc_resolutions_fulltext_data_11May26.csv",
                        usecols=["ID_PAT", "full_text"])

# Drop rows where full_text is missing or blank
before = len(fulltexts)
fulltexts = fulltexts[fulltexts["full_text"].notna() & (fulltexts["full_text"].str.strip() != "")]
after = len(fulltexts)

print(f"Full texts total:            {before:,}")
print(f"Full texts with actual text: {after:,}  (dropped {before - after:,} empty)")
fulltexts.head()

## 3. Merge — keep only rulings that have a summary

In [ ]:
matched = rulings.merge(fulltexts, on="ID_PAT", how="inner")

print(f"Rulings with a full text:    {len(matched):,}")
print(f"Rulings WITHOUT a full text: {len(rulings) - len(matched):,}")
matched.head()

## 4. Random sample

In [ ]:
n = min(SAMPLE_SIZE, len(matched))
sample = matched.sample(n=n, random_state=RANDOM_SEED).reset_index(drop=True)

print(f"Sample size: {len(sample)}")
print(f"\nProcedure distribution in sample:")
print(sample["procedure"].value_counts().to_string())
print(f"\nYear range: {sample['year_r'].min()} – {sample['year_r'].max()}")
sample

## 5. Save

In [ ]:
sample.to_csv("random_sample_handcode.csv", index=False)
print("Saved → random_sample_handcode.csv")

## 6. Extend the sample by 50 new observations

In [ ]:
EXTENSION_SIZE = 50
RANDOM_SEED_EXT = 123   # different seed from the original draw

# Load the already-coded file — preserves all hand-coded labels
# (sep=";" because Excel saved the file with semicolon delimiter)
existing = pd.read_csv("random_sample_handcode.csv", sep=";")
print(f"Existing coded observations: {len(existing)}")
print(f"Columns: {existing.columns.tolist()}")
existing.head()

In [ ]:
# Rebuild the eligible pool independently — safe to run without re-running cells 1-5,
# which would overwrite the file and lose hand-coded labels.
_main = pd.read_csv("complete_data_set7May2024.csv",
                    usecols=["ID_PAT", "procedure", "year_r"])
_rulings = _main.drop_duplicates(subset="ID_PAT").reset_index(drop=True)

_fulltexts = pd.read_csv("tc_resolutions_fulltext_data_11May26.csv",
                         usecols=["ID_PAT", "full_text"])
_fulltexts = _fulltexts[
    _fulltexts["full_text"].notna() & (_fulltexts["full_text"].str.strip() != "")
]

_matched = _rulings.merge(_fulltexts, on="ID_PAT", how="inner")
print(f"Eligible pool: {len(_matched):,} rulings")

# Exclude already-sampled IDs
already_sampled = set(existing["ID_PAT"])
pool = _matched[~_matched["ID_PAT"].isin(already_sampled)]
print(f"Pool after excluding existing sample: {len(pool):,} rulings")

# Draw the extension
n_ext = min(EXTENSION_SIZE, len(pool))
extension = pool.sample(n=n_ext, random_state=RANDOM_SEED_EXT).reset_index(drop=True)
print(f"\nExtension size: {len(extension)}")
print(f"\nProcedure distribution in extension:")
print(extension["procedure"].value_counts().to_string())
print(f"\nYear range: {extension['year_r'].min()} – {extension['year_r'].max()}")
extension.head()

In [ ]:
# Combine: existing rows (with coded labels) + new rows (labels will be blank)
combined = pd.concat([existing, extension], ignore_index=True)
print(f"Combined sample: {len(combined)} rows  ({len(existing)} coded + {len(extension)} new)")

# Verify no duplicates
assert combined["ID_PAT"].nunique() == len(combined), "Duplicate ID_PAT found!"
print("No duplicate ID_PAT — OK")

combined.to_csv("random_sample_handcode.csv", index=False, sep=";")
print("Saved → random_sample_handcode.csv")

## 7. Stratified extension to 150 — oversample Abstract review

The original 100-case draw was a pure random sample, which faithfully reflects the population
(~93 % Concrete, ~7 % Abstract). However, 7 abstract-review cases provide insufficient
statistical power to evaluate classifier performance on the case type that is most substantively
central to the research question.

**Sampling design:**
- Target: 150 cases total — 110 Concrete + 40 Abstract.
- Already coded: 93 Concrete + 7 Abstract (kept intact, labels preserved).
- New draws: 17 Concrete (`RANDOM_SEED_STRAT_C = 789`) + 33 Abstract (`RANDOM_SEED_STRAT_A = 789`).
- Both draws exclude all IDs already present in the existing 100-case file.

**Transparency note:** Because abstract cases are overrepresented relative to their population
share (26.7 % in sample vs. 7.1 % in population), pooled accuracy metrics should be computed
as population-weighted averages. Stratum-specific metrics are reported separately.

In [6]:
RANDOM_SEED_STRAT_C = 789   # seed for the 17 new Concrete draws
RANDOM_SEED_STRAT_A = 789   # seed for the 33 new Abstract draws

TARGET_ABSTRACT  = 40
TARGET_CONCRETE  = 110

# ── Load the already-coded 100-case file ─────────────────────────────────────
existing = pd.read_csv("random_sample_handcode_100.csv", sep=";")
print(f"Existing coded observations: {len(existing)}")
print("Procedure breakdown in existing sample:")
print(existing["procedure"].value_counts().to_string())

n_existing_abstract = (existing["procedure"] == "Abstract").sum()
n_existing_concrete = (existing["procedure"] == "Concrete").sum()

n_new_abstract = TARGET_ABSTRACT - n_existing_abstract
n_new_concrete = TARGET_CONCRETE - n_existing_concrete
print(f"\nNew draws needed — Abstract: {n_new_abstract}, Concrete: {n_new_concrete}")

# ── Rebuild the eligible pool ─────────────────────────────────────────────────
_main = pd.read_csv("complete_data_set7May2024.csv", usecols=["ID_PAT", "procedure", "year_r"])
_rulings = _main.drop_duplicates(subset="ID_PAT").reset_index(drop=True)

_fulltexts = pd.read_csv("tc_resolutions_fulltext_data_11May26.csv", usecols=["ID_PAT", "full_text"])
_fulltexts = _fulltexts[
    _fulltexts["full_text"].notna() & (_fulltexts["full_text"].str.strip() != "")
]
_matched = _rulings.merge(_fulltexts, on="ID_PAT", how="inner")

# Exclude IDs already in the coded sample
already_sampled = set(existing["ID_PAT"])
pool = _matched[~_matched["ID_PAT"].isin(already_sampled)].copy()

print(f"\nEligible pool (excluding existing sample): {len(pool):,} rulings")
print("Pool procedure breakdown:")
print(pool["procedure"].value_counts().to_string())

# ── Stratified draws ──────────────────────────────────────────────────────────
pool_abstract = pool[pool["procedure"] == "Abstract"]
pool_concrete = pool[pool["procedure"] == "Concrete"]

new_abstract = pool_abstract.sample(n=n_new_abstract, random_state=RANDOM_SEED_STRAT_A).reset_index(drop=True)
new_concrete = pool_concrete.sample(n=n_new_concrete, random_state=RANDOM_SEED_STRAT_C).reset_index(drop=True)

print(f"\nNew Abstract drawn: {len(new_abstract)}")
print(f"New Concrete drawn: {len(new_concrete)}")

# ── Combine: existing (with labels) + new rows (labels blank) ─────────────────
extension = pd.concat([new_abstract, new_concrete], ignore_index=True)
# Drop full_text to keep the file light (same convention as existing file)
extension = extension.drop(columns=["full_text"], errors="ignore")

combined = pd.concat([existing, extension], ignore_index=True)

assert combined["ID_PAT"].nunique() == len(combined), "Duplicate ID_PAT found!"
print(f"\nCombined sample: {len(combined)} rows  "
      f"({len(existing)} coded + {len(extension)} new to code)")
print("\nFinal procedure distribution:")
print(combined["procedure"].value_counts().to_string())
print(f"\nYear range: {combined['year_r'].min()} – {combined['year_r'].max()}")

combined.to_csv("random_sample_handcode_150.csv", index=False, sep=";")
print("\nSaved → random_sample_handcode_150.csv")

Existing coded observations: 100
Procedure breakdown in existing sample:
procedure
Concrete    93
Abstract     7

New draws needed — Abstract: 33, Concrete: 17

Eligible pool (excluding existing sample): 18,848 rulings
Pool procedure breakdown:
procedure
Concrete    17245
Abstract     1603

New Abstract drawn: 33
New Concrete drawn: 17

Combined sample: 150 rows  (100 coded + 50 new to code)

Final procedure distribution:
procedure
Concrete    110
Abstract     40

Year range: 1981 – 2023

Saved → random_sample_handcode_150.csv
